# Layer 1 — Volatility LightGBM (Chapter 4.2)

**Thesis section.** 4.2 — volatility-regime forecaster, LightGBM with block+daily features

**Inputs.** `data/vol_data/<TICKER>_vol_{train,val,test}.csv`

**Outputs.** `results/vol_prediction_v3_results.json`, per-ticker vol_lstm checkpoints under `checkpoints/vol_lstm_<TICKER>.pt`

**Expected runtime.** 10–20 min. **Expected GPU.** optional.

> All paths in the CONFIG cell below resolve relative to the repo root. On
> Google Colab, uncomment the Drive fallback line.


In [ ]:
# === CONFIG (edit paths here) ===
from pathlib import Path

CONFIG = {
    "data_dir":        Path("../../data"),         # processed + features + splits
    "results_dir":     Path("../../results"),
    "checkpoints_dir": Path("../../checkpoints"),
    "seed":            42,
    "device":          "cuda",                      # or "cpu"
    # Colab fallback — uncomment if running on Colab with the dataset mounted:
    # "data_dir": Path("/content/drive/MyDrive/thesis_data"),
}
for key, path in CONFIG.items():
    if isinstance(path, Path):
        path.mkdir(parents=True, exist_ok=True)


In [ ]:
"""
Turning Point Detection + LSTM Direction Prediction
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import time
import os
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# 1. Configuration
# ============================================================
class Config:
    # Data
    TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
    FREQ_RAW = "1min"
    FREQ_PRED = "15min"  # prediction timeframe
    DRIVE_BASE = CONFIG['data_dir'] / r'splits'

    # Turning point detection (on 1-min data)
    TP_REVERSAL_PCT = 1.0    # minimum reversal % to confirm turning point
    TP_MIN_DURATION = 60     # minimum bars (minutes) between turning points

    # 15-min resampling
    RESAMPLE_PERIOD = 15     # minutes per bar

    # LSTM
    SEQ_LEN = 30             # 30 × 15min = 7.5 hours of history
    HIDDEN_SIZE = 64
    NUM_LAYERS = 2
    DROPOUT = 0.2
    BATCH_SIZE = 32
    EPOCHS = 80
    LR = 5e-4
    PATIENCE = 15

    # Prediction target
    HORIZONS = [1, 2, 3, 4]  # sweep multiple horizons
    MIN_MOVE_PCT = 0.15      # minimum % move to count as directional (filter flat)

    # TP parameter sweep
    TP_SWEEP = [
        (0.5, 30),
        (0.75, 45),
        (1.0, 60),
        (1.5, 90),
    ]

    # Device & seed
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED = 42


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(Config.SEED)
print(f"Device: {Config.DEVICE}")


# ============================================================
# 2. Data Loading
# ============================================================
def load_data(config):
    """Load 1-min data from pre-split CSVs."""
    data_dir = f"{config.DRIVE_BASE}/{config.TICKER}_{config.FREQ_RAW}"
    print(f"\nLoading data from: {data_dir}")

    dfs = []
    for split in ['train', 'val', 'test']:
        path = f"{data_dir}/{split}.csv"
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['split'] = split
            dfs.append(df)
            print(f"  {split}: {len(df):,} rows")

    df = pd.concat(dfs, ignore_index=True)

    # Parse timestamp
    if 'ts_event' in df.columns:
        df['timestamp'] = pd.to_datetime(df['ts_event'])
    elif 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    else:
        # Try first column
        df['timestamp'] = pd.to_datetime(df.iloc[:, 0])

    df = df.sort_values('timestamp').reset_index(drop=True)

    # Standardize column names
    col_map = {}
    for col in df.columns:
        cl = col.lower()
        if cl in ['open', 'high', 'low', 'close', 'volume']:
            col_map[col] = cl
    df = df.rename(columns=col_map)

    # Ensure numeric
    for col in ['open', 'high', 'low', 'close', 'volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Mark train/test boundary
    train_val_mask = df['split'].isin(['train', 'val'])
    train_end_ts = df.loc[train_val_mask, 'timestamp'].max()

    print(f"  Total: {len(df):,} rows")
    print(f"  Range: {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Train+Val ends: {train_end_ts}")

    return df, train_end_ts


# ============================================================
# 3. Causal Turning Point Detection (1-min)
# ============================================================
def detect_turning_points_causal(prices, config, timestamps=None):
    """
    Strictly causal turning point detection (v3 - with overnight gap filtering).

    Uses a zigzag approach: alternates between seeking tops and bottoms.
    A reversal is confirmed the MOMENT price moves rev_pct% from the
    running extreme.

    If timestamps are provided, resets state at overnight boundaries
    to avoid detecting false TPs caused by overnight gaps.
    """
    n = len(prices)
    tp_events = []

    rev_pct = config.TP_REVERSAL_PCT / 100.0
    min_dur = config.TP_MIN_DURATION

    # Detect overnight boundaries (gap > 2 hours between consecutive bars)
    overnight = set()
    if timestamps is not None:
        ts = pd.DatetimeIndex(timestamps)
        gaps = np.diff(ts.astype(np.int64)) / 1e9  # gap in seconds
        overnight = set(np.where(gaps > 7200)[0] + 1)  # 2 hours = 7200s

    # State
    running_max = prices[0]
    running_max_idx = 0
    running_min = prices[0]
    running_min_idx = 0
    last_tp_idx = -min_dur
    seeking = 'both'

    for t in range(1, n):
        p = prices[t]

        # Reset state at overnight boundary
        if t in overnight:
            running_max = p
            running_max_idx = t
            running_min = p
            running_min_idx = t
            # Don't reset seeking or last_tp_idx — just the extremes
            continue

        # Update running extremes
        if p > running_max:
            running_max = p
            running_max_idx = t
        if p < running_min:
            running_min = p
            running_min_idx = t

        # --- Seek TOP: price dropped rev_pct from running_max ---
        if seeking in ('both', 'top'):
            if running_max > 0 and (running_max - p) / running_max >= rev_pct:
                if t - last_tp_idx >= min_dur:
                    tp_events.append({
                        'idx': running_max_idx,
                        'confirm_idx': t,
                        'type': 'top',
                        'price': running_max
                    })
                    last_tp_idx = t
                    seeking = 'bottom'
                    running_min = p
                    running_min_idx = t
                    running_max = p
                    running_max_idx = t
                    continue

        # --- Seek BOTTOM: price rose rev_pct from running_min ---
        if seeking in ('both', 'bottom'):
            if running_min > 0 and (p - running_min) / running_min >= rev_pct:
                if t - last_tp_idx >= min_dur:
                    tp_events.append({
                        'idx': running_min_idx,
                        'confirm_idx': t,
                        'type': 'bottom',
                        'price': running_min
                    })
                    last_tp_idx = t
                    seeking = 'top'
                    running_max = p
                    running_max_idx = t
                    running_min = p
                    running_min_idx = t
                    continue

    return tp_events


# ============================================================
# 4. Resample to 15-min & Build Features
# ============================================================
def resample_to_15min(df, config):
    """Resample 1-min OHLCV to 15-min bars."""
    df_ts = df.set_index('timestamp')

    rule = f'{config.RESAMPLE_PERIOD}min'
    ohlcv = df_ts.resample(rule).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna()

    ohlcv = ohlcv.reset_index()
    print(f"  Resampled: {len(df):,} (1min) → {len(ohlcv):,} (15min)")
    return ohlcv


def map_tp_to_15min(tp_events, df_1min, df_15min):
    """
    Map turning point confirmations to 15-min bar indices.

    For each TP event, find which 15-min bar contains the CONFIRMATION time.
    This ensures causality: the 15-min bar knows about the TP only after confirmation.
    """
    # Get timestamps
    ts_1min = df_1min['timestamp'].values
    ts_15min = df_15min['timestamp'].values

    tp_15min_indices = []
    for tp in tp_events:
        confirm_ts = ts_1min[tp['confirm_idx']]
        # Find the 15-min bar that contains or follows this confirmation
        bar_idx = np.searchsorted(ts_15min, confirm_ts, side='right') - 1
        if 0 <= bar_idx < len(ts_15min):
            tp_15min_indices.append({
                'bar_idx': bar_idx,
                'type': tp['type'],
                'price': tp['price'],
                'confirm_ts': confirm_ts,
                'tp_idx_1min': tp['idx'],
                'confirm_idx_1min': tp['confirm_idx']
            })

    return tp_15min_indices


def build_features(df_15min, tp_15min_events, config):
    """
    Build feature matrix for 15-min bars.

    Features:
      1-4:  Price-based: return, log_return, high_low_range, body_ratio
      5-8:  Moving averages: SMA_5, SMA_20 ratios
      9-12: Volume: RVOL, volume_change
      13-16: Volatility: rolling_std_5, rolling_std_20, ATR
      17-20: Turning point features:
             - bars_since_last_tp
             - last_tp_type (1=bottom/buy, -1=top/sell, 0=none)
             - reversal_magnitude (% move since last TP)
             - trend_strength (cumulative return since TP)
    """
    close = df_15min['close'].values.astype(float)
    open_ = df_15min['open'].values.astype(float)
    high = df_15min['high'].values.astype(float)
    low = df_15min['low'].values.astype(float)
    volume = df_15min['volume'].values.astype(float)
    n = len(df_15min)

    features = {}

    # --- Price features ---
    returns = np.zeros(n)
    returns[1:] = (close[1:] - close[:-1]) / np.maximum(close[:-1], 1e-8)
    features['return'] = returns

    log_ret = np.zeros(n)
    log_ret[1:] = np.log(np.maximum(close[1:], 1e-8)) - np.log(np.maximum(close[:-1], 1e-8))
    features['log_return'] = log_ret

    hl_range = (high - low) / np.maximum(close, 1e-8)
    features['hl_range'] = hl_range

    body = np.abs(close - open_) / np.maximum(high - low, 1e-8)
    features['body_ratio'] = np.nan_to_num(body, nan=0.5)

    # Upper/lower shadow ratios
    upper_shadow = (high - np.maximum(close, open_)) / np.maximum(high - low, 1e-8)
    lower_shadow = (np.minimum(close, open_) - low) / np.maximum(high - low, 1e-8)
    features['upper_shadow'] = np.nan_to_num(upper_shadow, nan=0.0)
    features['lower_shadow'] = np.nan_to_num(lower_shadow, nan=0.0)

    # --- Moving average features ---
    close_s = pd.Series(close)
    sma5 = close_s.rolling(5, min_periods=1).mean().values
    sma20 = close_s.rolling(20, min_periods=1).mean().values
    features['price_sma5_ratio'] = close / np.maximum(sma5, 1e-8) - 1
    features['price_sma20_ratio'] = close / np.maximum(sma20, 1e-8) - 1
    features['sma5_sma20_ratio'] = sma5 / np.maximum(sma20, 1e-8) - 1

    # --- Volume features ---
    vol_s = pd.Series(volume)
    vol_sma20 = vol_s.rolling(20, min_periods=1).mean().values
    features['rvol'] = np.log1p(volume / np.maximum(vol_sma20, 1))
    vol_change = np.zeros(n)
    vol_change[1:] = (volume[1:] - volume[:-1]) / np.maximum(volume[:-1], 1)
    features['vol_change'] = vol_change

    # --- Volatility features ---
    ret_s = pd.Series(returns)
    features['vol_5'] = ret_s.rolling(5, min_periods=1).std().values
    features['vol_20'] = ret_s.rolling(20, min_periods=1).std().values

    # ATR
    tr = np.maximum(high - low,
                    np.maximum(np.abs(high - np.roll(close, 1)),
                               np.abs(low - np.roll(close, 1))))
    tr[0] = high[0] - low[0]
    features['atr'] = pd.Series(tr).rolling(14, min_periods=1).mean().values / np.maximum(close, 1e-8)

    # --- RSI ---
    gain = np.maximum(returns, 0)
    loss = np.maximum(-returns, 0)
    avg_gain = pd.Series(gain).rolling(14, min_periods=1).mean().values
    avg_loss = pd.Series(loss).rolling(14, min_periods=1).mean().values
    rs = avg_gain / np.maximum(avg_loss, 1e-10)
    features['rsi'] = 1 - 1 / (1 + rs)  # normalized to [0, 1]

    # --- Turning point features (CAUSAL) ---
    bars_since_tp = np.full(n, 999.0)  # large number if no TP
    last_tp_type = np.zeros(n)          # 0=none, 1=bottom, -1=top
    reversal_mag = np.zeros(n)          # % move since TP price
    tp_signal = np.zeros(n)             # 1 on bars with TP confirmation

    # Sort TP events by bar_idx
    sorted_tps = sorted(tp_15min_events, key=lambda x: x['bar_idx'])

    tp_ptr = 0
    current_tp = None

    for t in range(n):
        # Check if any TP was confirmed at or before this bar
        while tp_ptr < len(sorted_tps) and sorted_tps[tp_ptr]['bar_idx'] <= t:
            current_tp = sorted_tps[tp_ptr]
            if sorted_tps[tp_ptr]['bar_idx'] == t:
                tp_signal[t] = 1
            tp_ptr += 1

        if current_tp is not None:
            bars_since_tp[t] = t - current_tp['bar_idx']
            last_tp_type[t] = 1.0 if current_tp['type'] == 'bottom' else -1.0
            reversal_mag[t] = (close[t] - current_tp['price']) / current_tp['price']

    features['bars_since_tp'] = bars_since_tp / 100.0  # normalize
    features['last_tp_type'] = last_tp_type
    features['reversal_mag'] = reversal_mag
    features['tp_signal'] = tp_signal

    feature_df = pd.DataFrame(features)
    return feature_df


# ============================================================
# 5. Build Training Samples
# ============================================================
def build_classification_dataset(feature_df, df_15min, tp_15min_events, config, horizon=None):
    """Build (X, y) for binary classification."""
    close = df_15min['close'].values.astype(float)
    n = len(close)
    seq_len = config.SEQ_LEN
    if horizon is None:
        horizon = config.HORIZONS[0]
    min_move = config.MIN_MOVE_PCT / 100.0

    feature_matrix = feature_df.values.astype(np.float32)
    n_features = feature_matrix.shape[1]

    # Collect samples
    X_list = []
    y_list = []
    is_tp_list = []  # whether this sample is a TP event
    timestamps = []

    # Create TP bar set for quick lookup
    tp_bars = {}
    for tp in tp_15min_events:
        tp_bars[tp['bar_idx']] = tp

    # For every valid bar (enough history + room for horizon)
    for t in range(seq_len, n - horizon):
        # Future return
        future_ret = (close[t + horizon] - close[t]) / close[t]

        # Skip flat moves (de minimis filter applied at TRAINING time)
        if abs(future_ret) < min_move:
            continue

        # Direction label
        direction = 1 if future_ret > 0 else 0

        # If this is a TP bar, label is "did the TP signal correctly predict direction?"
        if t in tp_bars:
            tp = tp_bars[t]
            if tp['type'] == 'bottom':
                # Bottom TP predicts UP
                label = 1 if future_ret > 0 else 0
            else:
                # Top TP predicts DOWN
                label = 1 if future_ret < 0 else 0
            is_tp = True
        else:
            # Non-TP bar: simple direction label
            label = direction
            is_tp = False

        # Extract sequence
        seq = feature_matrix[t - seq_len + 1:t + 1]  # (seq_len, n_features)

        X_list.append(seq)
        y_list.append(label)
        is_tp_list.append(is_tp)
        timestamps.append(df_15min['timestamp'].iloc[t])

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    is_tp = np.array(is_tp_list)
    timestamps = np.array(timestamps)

    return X, y, is_tp, timestamps


# ============================================================
# 6. LSTM Classifier
# ============================================================
class LSTMClassifier(nn.Module):
    """LSTM for binary direction classification."""

    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        logit = self.fc(last_hidden)
        return logit.squeeze(-1)


# ============================================================
# 7. Training
# ============================================================
def train_classifier(X_train, y_train, X_val, y_val,
                     is_tp_train, config):
    """
    Train LSTM classifier with class-weighted + TP-weighted loss.
    """
    n_features = X_train.shape[2]
    model = LSTMClassifier(
        input_size=n_features,
        hidden_size=config.HIDDEN_SIZE,
        num_layers=config.NUM_LAYERS,
        dropout=config.DROPOUT
    ).to(config.DEVICE)

    # Class weights (handle imbalance)
    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(config.DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Sample weights: TP samples get higher weight
    sample_weights = np.ones(len(y_train), dtype=np.float32)
    sample_weights[is_tp_train] = 3.0  # TP samples 3x weight

    optimizer = torch.optim.Adam(model.parameters(), lr=config.LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # DataLoaders
    train_ds = TensorDataset(
        torch.FloatTensor(X_train),
        torch.FloatTensor(y_train.astype(np.float32)),
        torch.FloatTensor(sample_weights)
    )
    val_ds = TensorDataset(
        torch.FloatTensor(X_val),
        torch.FloatTensor(y_val.astype(np.float32))
    )
    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE)

    # Training loop
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(config.EPOCHS):
        model.train()
        train_loss = 0
        for X_b, y_b, w_b in train_loader:
            X_b = X_b.to(config.DEVICE)
            y_b = y_b.to(config.DEVICE)
            w_b = w_b.to(config.DEVICE)

            logits = model(X_b)
            loss = nn.BCEWithLogitsLoss(
                pos_weight=pos_weight,
                weight=w_b
            )(logits, y_b)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(y_b)

        train_loss /= len(train_ds)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                X_b = batch[0].to(config.DEVICE)
                y_b = batch[1].to(config.DEVICE)
                logits = model(X_b)
                loss = criterion(logits, y_b)
                val_loss += loss.item() * len(y_b)
        val_loss /= len(val_ds)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{config.EPOCHS}: "
                  f"train={train_loss:.4f}, val={val_loss:.4f}")

        if patience_counter >= config.PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"    Best val loss: {best_val_loss:.4f}")
    return model


# ============================================================
# 8. Evaluation
# ============================================================
def evaluate_model(model, X_test, y_test, is_tp_test, timestamps_test,
                   df_15min_test_close, config):
    """
    Evaluate on test set with separate metrics for TP and non-TP bars.
    """
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X_test).to(config.DEVICE)
        logits = model(X_t).cpu().numpy()
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)

    print(f"\n{'='*60}")
    print(f"  EVALUATION RESULTS")
    print(f"{'='*60}")

    # Overall metrics
    acc = np.mean(preds == y_test) * 100
    print(f"\n  Overall: Acc={acc:.1f}% (N={len(y_test)})")
    print(f"  Class distribution: {(y_test==1).sum()} pos, {(y_test==0).sum()} neg")
    print(f"  Prediction distribution: {(preds==1).sum()} pos, {(preds==0).sum()} neg")

    # TP-only metrics (the important ones)
    if is_tp_test.sum() > 0:
        tp_mask = is_tp_test
        tp_acc = np.mean(preds[tp_mask] == y_test[tp_mask]) * 100
        tp_n = tp_mask.sum()

        # Statistical significance
        from scipy import stats
        z = (tp_acc/100 - 0.5) / np.sqrt(0.25 / tp_n)
        p_val = 2 * (1 - stats.norm.cdf(abs(z)))

        print(f"\n  === TURNING POINT SIGNALS ===")
        print(f"  TP Accuracy: {tp_acc:.1f}% (N={tp_n})")
        print(f"  Z-score: {z:.2f}, p-value: {p_val:.4f}")
        print(f"  {'*** SIGNIFICANT ***' if p_val < 0.05 else '(not significant)'}")

        # TP breakdown by type
        tp_indices = np.where(tp_mask)[0]
        # We need to trace back to the original TP events to get types
        # For now, just show overall TP accuracy

        # Confidence-based filtering
        print(f"\n  --- Confidence-based filtering ---")
        for threshold in [0.5, 0.55, 0.6, 0.65, 0.7]:
            high_conf = np.abs(probs - 0.5) > (threshold - 0.5)
            high_conf_tp = high_conf & tp_mask
            if high_conf_tp.sum() > 10:
                hc_acc = np.mean(preds[high_conf_tp] == y_test[high_conf_tp]) * 100
                print(f"    Prob > {threshold:.0%}: Acc={hc_acc:.1f}% (N={high_conf_tp.sum()})")

    # Non-TP metrics
    non_tp_mask = ~is_tp_test
    if non_tp_mask.sum() > 0:
        non_tp_acc = np.mean(preds[non_tp_mask] == y_test[non_tp_mask]) * 100
        print(f"\n  Non-TP bars: Acc={non_tp_acc:.1f}% (N={non_tp_mask.sum()})")

    # Simple backtest on TP signals only
    print(f"\n  === SIMPLE BACKTEST (TP signals only) ===")
    if is_tp_test.sum() > 0:
        # On TP bars: if model predicts 1 (trend continues), take the trade
        # After BOTTOM: go long if pred=1
        # After TOP: go short if pred=1
        # For simplicity: pred=1 means "agree with TP signal"
        tp_correct = preds[tp_mask] == y_test[tp_mask]
        tp_probs_correct = probs[tp_mask]

        # Win rate
        win_rate = np.mean(tp_correct) * 100
        # Average confidence on wins vs losses
        avg_conf_win = np.mean(tp_probs_correct[tp_correct]) if tp_correct.sum() > 0 else 0
        avg_conf_loss = np.mean(tp_probs_correct[~tp_correct]) if (~tp_correct).sum() > 0 else 0

        print(f"  Win rate: {win_rate:.1f}%")
        print(f"  Avg confidence (wins):   {avg_conf_win:.3f}")
        print(f"  Avg confidence (losses): {avg_conf_loss:.3f}")
        print(f"  Total trades: {tp_mask.sum()}")

    return {
        'overall_acc': acc,
        'tp_acc': tp_acc if is_tp_test.sum() > 0 else None,
        'tp_n': int(is_tp_test.sum()),
        'preds': preds,
        'probs': probs,
        'y_test': y_test,
        'is_tp': is_tp_test
    }


# ============================================================
# 9. Visualization
# ============================================================
def plot_turning_points(df_15min, tp_events, start_idx=0, end_idx=500):
    """Plot price with marked turning points."""
    fig, ax = plt.subplots(figsize=(15, 6))

    subset = df_15min.iloc[start_idx:end_idx]
    ax.plot(range(len(subset)), subset['close'].values, 'k-', linewidth=0.8, alpha=0.8)

    for tp in tp_events:
        idx = tp['bar_idx']
        if start_idx <= idx < end_idx:
            plot_idx = idx - start_idx
            color = 'red' if tp['type'] == 'top' else 'green'
            marker = 'v' if tp['type'] == 'top' else '^'
            ax.scatter(plot_idx, tp['price'], color=color, marker=marker,
                      s=100, zorder=5)

    ax.set_xlabel('15-min bar index')
    ax.set_ylabel('Close Price')
    ax.set_title(f'Turning Points Detection (bars {start_idx}-{end_idx})')
    ax.legend(['Close', 'Top', 'Bottom'])
    plt.tight_layout()
    return fig


def plot_results(results):
    """Plot prediction confidence distribution."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Confidence histogram
    ax = axes[0]
    probs_correct = results['probs'][results['preds'] == results['y_test']]
    probs_wrong = results['probs'][results['preds'] != results['y_test']]
    ax.hist(probs_correct, bins=30, alpha=0.5, label='Correct', color='green')
    ax.hist(probs_wrong, bins=30, alpha=0.5, label='Wrong', color='red')
    ax.axvline(x=0.5, color='black', linestyle='--')
    ax.set_xlabel('Prediction Probability')
    ax.set_ylabel('Count')
    ax.set_title('Confidence Distribution')
    ax.legend()

    # TP vs non-TP accuracy
    ax = axes[1]
    tp_mask = results['is_tp']
    categories = ['All', 'TP Signals', 'Non-TP']
    accs = [
        results['overall_acc'],
        results['tp_acc'] if results['tp_acc'] is not None else 0,
        np.mean(results['preds'][~tp_mask] == results['y_test'][~tp_mask]) * 100
        if (~tp_mask).sum() > 0 else 0
    ]
    counts = [len(results['y_test']), tp_mask.sum(), (~tp_mask).sum()]

    bars = ax.bar(categories, accs, color=['steelblue', 'forestgreen', 'gray'])
    ax.axhline(y=50, color='red', linestyle='--', label='Random')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'N={count}', ha='center', fontsize=9)
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by Signal Type')
    ax.set_ylim(40, 70)
    ax.legend()

    plt.tight_layout()
    return fig


# ============================================================
# 10b. Quick evaluate (returns dict, no printing)
# ============================================================
def quick_evaluate(model, X_test, y_test, is_tp_test, config):
    """Quick evaluation returning key metrics."""
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X_test).to(config.DEVICE)
        # Process in batches to avoid OOM
        logits_list = []
        for i in range(0, len(X_t), 512):
            batch = X_t[i:i+512]
            logits_list.append(model(batch).cpu().numpy())
        logits = np.concatenate(logits_list)
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    overall_acc = np.mean(preds == y_test) * 100
    tp_mask = is_tp_test

    result = {'overall_acc': overall_acc, 'N': len(y_test)}

    if tp_mask.sum() > 0:
        tp_acc = np.mean(preds[tp_mask] == y_test[tp_mask]) * 100
        tp_n = int(tp_mask.sum())
        from scipy import stats
        z = (tp_acc/100 - 0.5) / np.sqrt(0.25 / tp_n)
        p_val = 2 * (1 - stats.norm.cdf(abs(z)))
        result.update({'tp_acc': tp_acc, 'tp_n': tp_n, 'z': z, 'p': p_val})

        # High confidence
        for thr in [0.6, 0.7]:
            hc = np.abs(probs - 0.5) > (thr - 0.5)
            hc_tp = hc & tp_mask
            if hc_tp.sum() > 10:
                result[f'tp_acc_{int(thr*100)}'] = np.mean(preds[hc_tp] == y_test[hc_tp]) * 100
                result[f'tp_n_{int(thr*100)}'] = int(hc_tp.sum())
    else:
        result.update({'tp_acc': None, 'tp_n': 0, 'z': 0, 'p': 1.0})

    return result


# ============================================================
# 11. Main with sweep
# ============================================================
def load_ticker_data(ticker, config):
    """Load and preprocess a single ticker's data. Returns None if not found."""
    data_dir = f"{config.DRIVE_BASE}/{ticker}_{config.FREQ_RAW}"
    if not os.path.exists(data_dir):
        return None

    dfs = []
    for split in ['train', 'val', 'test']:
        path = f"{data_dir}/{split}.csv"
        if os.path.exists(path):
            df_part = pd.read_csv(path)
            df_part['split'] = split
            dfs.append(df_part)

    if not dfs:
        return None

    df = pd.concat(dfs, ignore_index=True)

    if 'ts_event' in df.columns:
        df['timestamp'] = pd.to_datetime(df['ts_event'])
    elif 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    else:
        df['timestamp'] = pd.to_datetime(df.iloc[:, 0])

    df = df.sort_values('timestamp').reset_index(drop=True)

    col_map = {}
    for col in df.columns:
        cl = col.lower()
        if cl in ['open', 'high', 'low', 'close', 'volume']:
            col_map[col] = cl
    df = df.rename(columns=col_map)

    for col in ['open', 'high', 'low', 'close', 'volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    train_val_mask = df['split'].isin(['train', 'val'])
    train_end_ts = df.loc[train_val_mask, 'timestamp'].max()

    return {'df': df, 'train_end_ts': train_end_ts}


# All functions defined. Ready for CNN_TP_Filter.
print('All TP pipeline functions loaded.')
print(f'Device: {Config.DEVICE}')

Device: cuda
All TP pipeline functions loaded.
Device: cuda


In [ ]:
"""
Dual CNN: Separate Bottom + Top Detectors
==========================================
CNN-Bottom: trained ONLY on downtrend samples → detect V bottoms
CNN-Top: trained ONLY on uptrend samples → detect ^ tops

Each CNN sees only its own trend type, no interference.
Top CNN gets extra features tuned for exhaustion patterns.

Paste after TP_functions_only cell + CNN_TrendRev_v2 functions.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# Config
# ============================================================
class DualCNNConfig:
    TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
    FREQ_RAW = "1min"
    DRIVE_BASE = CONFIG['data_dir'] / r'splits'
    RESAMPLE_PERIOD = 15

    TREND_PCT = 0.5
    TREND_LOOKBACK = 6
    REV_PCT = 0.5
    LOOKAHEAD = 6

    WINDOW = 30
    N_FEATURES = 16       # expanded for top-specific features
    CNN_EPOCHS = 80
    CNN_LR = 1e-3
    CNN_BATCH = 64
    CNN_PATIENCE = 15

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED = 42


# ============================================================
# 1. Features — 16 channels with top-specific additions
# ============================================================
def build_features_dual(df_15min):
    close = df_15min['close'].values.astype(float)
    open_ = df_15min['open'].values.astype(float)
    high = df_15min['high'].values.astype(float)
    low = df_15min['low'].values.astype(float)
    volume = df_15min['volume'].values.astype(float)
    n = len(close)

    # ── Base features (same as v2) ──
    ret = np.zeros(n)
    ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-8)
    hl_range = (high - low) / (close + 1e-8)
    body = (close - open_) / (close + 1e-8)
    vol_ma20 = pd.Series(volume).rolling(20, min_periods=1).mean().values
    vol_ratio = np.log1p(np.clip(volume / (vol_ma20 + 1e-8), 0, 10))
    position = (close - low) / (high - low + 1e-8)
    upper_shadow = (high - np.maximum(close, open_)) / (high - low + 1e-8)
    lower_shadow = (np.minimum(close, open_) - low) / (high - low + 1e-8)
    vol_5 = pd.Series(ret).rolling(5, min_periods=1).std().values

    # RSI
    delta = pd.Series(close).diff()
    gain = delta.where(delta > 0, 0.0).rolling(14, min_periods=1).mean().values
    loss_val = (-delta.where(delta < 0, 0.0)).rolling(14, min_periods=1).mean().values
    rs = gain / (loss_val + 1e-8)
    rsi = (100 - 100 / (1 + rs)) / 100.0

    # Volume divergence
    vol_change = np.zeros(n)
    vol_change[1:] = (volume[1:] - volume[:-1]) / (volume[:-1] + 1e-8)
    vol_divergence = -ret * vol_change

    # Momentum deceleration
    mom_5 = pd.Series(close).pct_change(5).values
    mom_10 = pd.Series(close).pct_change(10).values
    mom_decel = np.zeros(n)
    mom_decel[5:] = mom_5[5:] - mom_10[5:] / 2

    # BB position
    bb_ma = pd.Series(close).rolling(20, min_periods=1).mean().values
    bb_std = pd.Series(close).rolling(20, min_periods=1).std().values
    bb_pos = np.clip((close - (bb_ma - 2 * bb_std)) / (4 * bb_std + 1e-8), -0.5, 1.5)

    # Cumulative return
    cum_ret_6 = pd.Series(close).pct_change(6).values

    vol_change_clean = np.nan_to_num(vol_change, nan=0.0)

    # ── NEW: Top-specific features ──

    # Volume trend (declining volume = exhaustion at top)
    vol_ma5 = pd.Series(volume).rolling(5, min_periods=1).mean().values
    vol_trend = np.zeros(n)
    vol_trend[1:] = (vol_ma5[1:] - vol_ma5[:-1]) / (vol_ma5[:-1] + 1e-8)

    # Consecutive up/down bars (losing momentum)
    consec = np.zeros(n)
    for i in range(1, n):
        if ret[i] > 0 and consec[i-1] > 0:
            consec[i] = consec[i-1] + 1
        elif ret[i] < 0 and consec[i-1] < 0:
            consec[i] = consec[i-1] - 1
        elif ret[i] > 0:
            consec[i] = 1
        elif ret[i] < 0:
            consec[i] = -1
    consec_norm = np.clip(consec / 10.0, -1, 1)

    features = np.stack([
        ret, hl_range, body, vol_ratio,
        position, upper_shadow, lower_shadow, vol_5,
        rsi, vol_divergence, mom_decel, bb_pos,
        cum_ret_6, vol_change_clean,
        vol_trend, consec_norm
    ], axis=1)  # (n, 16)

    return np.nan_to_num(features, nan=0.0).astype(np.float32)


# ============================================================
# 2. Find trends and label (same logic as v2)
# ============================================================
def find_trend_and_label(df_15min, config):
    close = df_15min['close'].values.astype(float)
    n = len(close)
    trend_pct = config.TREND_PCT / 100.0
    rev_pct = config.REV_PCT / 100.0
    LB = config.TREND_LOOKBACK
    LA = config.LOOKAHEAD

    positions = []
    for t in range(LB, n - LA):
        p_now = close[t]
        if p_now <= 0:
            continue
        p_past = close[t - LB]
        if p_past <= 0:
            continue
        move = (p_now - p_past) / p_past

        if abs(move) < trend_pct:
            continue

        if move < -trend_pct:
            trend_dir = 'down'
            future_max = np.max(close[t+1:t+LA+1])
            reversal = (future_max - p_now) / p_now >= rev_pct
        elif move > trend_pct:
            trend_dir = 'up'
            future_min = np.min(close[t+1:t+LA+1])
            reversal = (p_now - future_min) / p_now >= rev_pct
        else:
            continue

        positions.append({
            'bar_idx': t,
            'trend_dir': trend_dir,
            'reversal': bool(reversal),
            'move_pct': move * 100
        })
    return positions


def build_dataset(features, positions, config):
    W = config.WINDOW
    n = len(features)
    X_list, y_list = [], []
    for pos in positions:
        t = pos['bar_idx']
        if t - W < 0 or t >= n:
            continue
        X_list.append(features[t - W:t])
        y_list.append(1 if pos['reversal'] else 0)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)


# ============================================================
# 3. Focal Loss
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none')
        pt = targets * torch.sigmoid(logits) + (1 - targets) * (1 - torch.sigmoid(logits))
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


# ============================================================
# 4. CNN Model
# ============================================================
class SpecializedCNN(nn.Module):
    def __init__(self, n_features=16, window=30):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 32, 5, padding=2),
            nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(4),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4, 64), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.head(self.conv(x.permute(0, 2, 1))).squeeze(-1)


# ============================================================
# 5. Train one specialized CNN
# ============================================================
def train_specialized(X_tr, y_tr, X_val, y_val, config, name=""):
    model = SpecializedCNN(config.N_FEATURES, config.WINDOW).to(config.DEVICE)

    n_pos = (y_tr == 1).sum()
    n_neg = (y_tr == 0).sum()
    pw = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(config.DEVICE)
    crit = FocalLoss(alpha=1.0, gamma=2.0, pos_weight=pw)

    opt = torch.optim.Adam(model.parameters(), lr=config.CNN_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', 0.5, patience=4)

    tds = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr.astype(np.float32)))
    vds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val.astype(np.float32)))
    tl = DataLoader(tds, batch_size=config.CNN_BATCH, shuffle=True)
    vl = DataLoader(vds, batch_size=config.CNN_BATCH)

    best_vl, pc, best_st = float('inf'), 0, None
    for ep in range(config.CNN_EPOCHS):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        vloss = 0
        with torch.no_grad():
            for xb, yb in vl:
                xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
                vloss += crit(model(xb), yb).item() * len(yb)
        vloss /= len(vds); sched.step(vloss)
        if vloss < best_vl:
            best_vl = vloss
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pc = 0
        else:
            pc += 1
        if (ep + 1) % 10 == 0:
            print(f"      [{name}] Epoch {ep+1}: val_loss={vloss:.4f}")
        if pc >= config.CNN_PATIENCE:
            print(f"      [{name}] Early stop at {ep+1}")
            break
    if best_st:
        model.load_state_dict(best_st)
        model = model.to(config.DEVICE)
    print(f"      [{name}] Best val_loss: {best_vl:.4f}")
    return model


# ============================================================
# 6. Predict
# ============================================================
def predict_probs(model, X, config):
    model.eval()
    with torch.no_grad():
        out = []
        for i in range(0, len(X), 1024):
            b = torch.FloatTensor(X[i:i+1024]).to(config.DEVICE)
            out.append(torch.sigmoid(model(b)).cpu().numpy())
    return np.concatenate(out)


# ============================================================
# 7. Fine-grained evaluation
# ============================================================
def eval_sweep(probs, y, base_rate, label=""):
    print(f"\n  {label}")
    print(f"  N={len(y)}, reversals={y.sum()} ({base_rate*100:.1f}%)")
    print(f"  {'thr':>5s}  {'signals':>7s}  {'prec':>6s}  {'recall':>6s}  {'lift':>5s}  {'z':>6s}")
    print(f"  {'─'*50}")

    for thr_x100 in range(45, 66):
        thr = thr_x100 / 100.0
        sig = probs > thr
        n_sig = sig.sum()
        if n_sig < 10:
            continue
        prec = np.mean(y[sig] == 1) * 100
        rec = np.mean(probs[y == 1] > thr) * 100 if (y == 1).sum() > 0 else 0
        lift = prec / (base_rate * 100) if base_rate > 0 else 0
        z = (prec / 100 - base_rate) / np.sqrt(base_rate * (1 - base_rate) / n_sig)
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        s = '***' if p < 0.05 else ''
        print(f"  {thr:5.2f}  {n_sig:7d}  {prec:5.1f}%  {rec:5.1f}%  {lift:4.2f}x  {z:+5.2f} {s}")


# ============================================================
# 8. Main
# ============================================================
def run_dual_cnn():
    config = DualCNNConfig()
    set_seed(config.SEED)

    print(f"\n{'='*70}")
    print(f"  DUAL CNN: Separate Bottom + Top Detectors")
    print(f"  Trend: ≥{config.TREND_PCT}% in {config.TREND_LOOKBACK} bars")
    print(f"  Reversal: ≥{config.REV_PCT}% in {config.LOOKAHEAD} bars ({config.LOOKAHEAD*15}min)")
    print(f"  Features: {config.N_FEATURES}, Window: {config.WINDOW}")
    print(f"{'='*70}")

    print(f"\n[1] Loading data...")

    # Separate collections for down and up
    down_tr_X, down_tr_y, down_v_X, down_v_y = [], [], [], []
    up_tr_X, up_tr_y, up_v_X, up_v_y = [], [], [], []
    test_data = {}

    for ticker in config.TICKERS:
        data = load_ticker_data(ticker, config)
        if data is None:
            continue
        df = data['df']
        train_end_ts = data['train_end_ts']

        df_15min = resample_to_15min(df, config)
        features = build_features_dual(df_15min)
        positions = find_trend_and_label(df_15min, config)

        if len(positions) < 50:
            continue

        train_mask = df_15min['timestamp'] <= train_end_ts
        train_end_idx = int(train_mask.sum())

        pos_train = [p for p in positions if p['bar_idx'] < train_end_idx]
        pos_test = [p for p in positions if p['bar_idx'] >= train_end_idx]

        # Split by direction
        pos_train_down = [p for p in pos_train if p['trend_dir'] == 'down']
        pos_train_up = [p for p in pos_train if p['trend_dir'] == 'up']
        pos_test_down = [p for p in pos_test if p['trend_dir'] == 'down']
        pos_test_up = [p for p in pos_test if p['trend_dir'] == 'up']

        # Build datasets per direction
        if len(pos_train_down) > 30:
            X_d, y_d = build_dataset(features, pos_train_down, config)
            vs = max(int(len(X_d) * 0.15), 1)
            down_tr_X.append(X_d[:-vs]); down_tr_y.append(y_d[:-vs])
            down_v_X.append(X_d[-vs:]); down_v_y.append(y_d[-vs:])

        if len(pos_train_up) > 30:
            X_u, y_u = build_dataset(features, pos_train_up, config)
            vs = max(int(len(X_u) * 0.15), 1)
            up_tr_X.append(X_u[:-vs]); up_tr_y.append(y_u[:-vs])
            up_v_X.append(X_u[-vs:]); up_v_y.append(y_u[-vs:])

        # Test data per direction
        if len(pos_test_down) > 10:
            X_td, y_td = build_dataset(features, pos_test_down, config)
            if ticker not in test_data:
                test_data[ticker] = {}
            test_data[ticker]['down'] = {'X': X_td, 'y': y_td}

        if len(pos_test_up) > 10:
            X_tu, y_tu = build_dataset(features, pos_test_up, config)
            if ticker not in test_data:
                test_data[ticker] = {}
            test_data[ticker]['up'] = {'X': X_tu, 'y': y_tu}

        n_d_tr = len(pos_train_down)
        n_u_tr = len(pos_train_up)
        n_d_te = len(pos_test_down)
        n_u_te = len(pos_test_up)
        print(f"  {ticker}: train(↓{n_d_tr} ↑{n_u_tr}), test(↓{n_d_te} ↑{n_u_te})")

    # Combine
    Xd_tr = np.concatenate(down_tr_X); yd_tr = np.concatenate(down_tr_y)
    Xd_v = np.concatenate(down_v_X); yd_v = np.concatenate(down_v_y)
    Xu_tr = np.concatenate(up_tr_X); yu_tr = np.concatenate(up_tr_y)
    Xu_v = np.concatenate(up_v_X); yu_v = np.concatenate(up_v_y)

    print(f"\n  Bottom CNN data: train={len(Xd_tr):,}, val={len(Xd_v):,}, "
          f"rev_rate={yd_tr.mean()*100:.1f}%")
    print(f"  Top CNN data:    train={len(Xu_tr):,}, val={len(Xu_v):,}, "
          f"rev_rate={yu_tr.mean()*100:.1f}%")

    # ── Train Bottom CNN ──
    print(f"\n[2] Training BOTTOM CNN (downtrend → V reversal)...")
    set_seed(config.SEED)
    bottom_cnn = train_specialized(Xd_tr, yd_tr, Xd_v, yd_v, config, "BOTTOM")

    # ── Train Top CNN ──
    print(f"\n[3] Training TOP CNN (uptrend → ^ reversal)...")
    set_seed(config.SEED)
    top_cnn = train_specialized(Xu_tr, yu_tr, Xu_v, yu_v, config, "TOP")

    # ── Validation ──
    print(f"\n[4] Validation:")
    probs_dv = predict_probs(bottom_cnn, Xd_v, config)
    probs_uv = predict_probs(top_cnn, Xu_v, config)
    eval_sweep(probs_dv, yd_v, yd_v.mean(), "VAL: Bottom CNN (downtrend)")
    eval_sweep(probs_uv, yu_v, yu_v.mean(), "VAL: Top CNN (uptrend)")

    # ── Test: Combined ──
    print(f"\n{'='*70}")
    print(f"  TEST RESULTS")
    print(f"{'='*70}")

    # Gather all test data per direction
    all_d_X, all_d_y = [], []
    all_u_X, all_u_y = [], []
    for ticker, td in test_data.items():
        if 'down' in td:
            all_d_X.append(td['down']['X']); all_d_y.append(td['down']['y'])
        if 'up' in td:
            all_u_X.append(td['up']['X']); all_u_y.append(td['up']['y'])

    Xd_te = np.concatenate(all_d_X); yd_te = np.concatenate(all_d_y)
    Xu_te = np.concatenate(all_u_X); yu_te = np.concatenate(all_u_y)

    probs_d = predict_probs(bottom_cnn, Xd_te, config)
    probs_u = predict_probs(top_cnn, Xu_te, config)

    eval_sweep(probs_d, yd_te, yd_te.mean(), "TEST: Bottom CNN — COMBINED (buy signals)")
    eval_sweep(probs_u, yu_te, yu_te.mean(), "TEST: Top CNN — COMBINED (sell signals)")

    # ── Per-ticker ──
    print(f"\n  ── PER-TICKER BREAKDOWN ──")
    for ticker, td in test_data.items():
        line = f"  {ticker:>6s}:"
        if 'down' in td:
            pd_ = predict_probs(bottom_cnn, td['down']['X'], config)
            yd_ = td['down']['y']
            for thr in [0.53, 0.55, 0.58, 0.60]:
                sig = pd_ > thr
                if sig.sum() > 5:
                    prec = np.mean(yd_[sig] == 1) * 100
                    line += f"  ↓{thr}={prec:.0f}%({sig.sum()})"
        if 'up' in td:
            pu_ = predict_probs(top_cnn, td['up']['X'], config)
            yu_ = td['up']['y']
            for thr in [0.53, 0.55, 0.58, 0.60]:
                sig = pu_ > thr
                if sig.sum() > 5:
                    prec = np.mean(yu_[sig] == 1) * 100
                    line += f"  ↑{thr}={prec:.0f}%({sig.sum()})"
        print(line)

    # ── Comparison vs shared model ──
    print(f"\n  ── COMPARISON ──")
    print(f"  Shared CNN v2 (downtrend thr=0.60): 78.7% (N=94)")
    print(f"  Shared CNN v2 (uptrend thr=0.58):   57.5% (N=313)")
    print(f"  Baseline: Attn-LSTM TP = 53.2% (N=2694)")


run_dual_cnn()


  DUAL CNN: Separate Bottom + Top Detectors
  Trend: ≥0.5% in 6 bars
  Reversal: ≥0.5% in 6 bars (90min)
  Features: 16, Window: 30

[1] Loading data...


KeyboardInterrupt: 

In [ ]:
"""
Enhanced Attn-LSTM with CNN + Volatility Features
===================================================
Idea: Feed CNN reversal probability + volatility prediction as extra
features to Attn-LSTM, so it can learn when to trust its own signal.

Pipeline:
1. Train CNN Bottom + Top models (from Dual CNN code)
2. Train volatility predictor (or use existing)
3. For each turning point, compute:
   - CNN reversal probability at that location
   - Volatility prediction
4. Append these as extra features to Attn-LSTM input
5. Compare DA with and without extra features

Paste after TP_functions_only + CNN Dual code cells.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Config
# ============================================================
class EnhancedConfig:
    TICKERS = ["AAPL", "MSFT", "GOOGL", "GOOG", "NVDA", "TSLA", "SPY", "QQQ"]
    FREQ_RAW = "1min"
    DRIVE_BASE = CONFIG['data_dir'] / r'splits'
    RESAMPLE_PERIOD = 15

    # Turning point params (same as baseline)
    REVERSAL_THR = 1.0
    MIN_DURATION = 60
    PRED_HORIZON = 45

    # CNN trend params (same as Dual CNN)
    TREND_PCT = 0.5
    TREND_LOOKBACK = 6
    REV_PCT = 0.5
    CNN_LOOKAHEAD = 6
    LOOKAHEAD = 6             # alias for find_trend_and_label
    CNN_WINDOW = 30
    WINDOW = 30               # alias for build_dataset
    CNN_N_FEATURES = 16
    N_FEATURES = 16           # alias for train_specialized

    # Volatility (Direct LSTM with HAR, adapted from V3)
    # Block size: 24 × 15min = 6 hours (same as V3's 12 × 30min)

    # Attn-LSTM
    LSTM_WINDOW = 30          # same as baseline
    LSTM_FEATURES_BASE = 5    # return, hl_range, body, vol_ratio, position
    LSTM_FEATURES_EXTRA = 4   # cnn_bottom_prob, cnn_top_prob, vol_current, vol_pred
    LSTM_HIDDEN = 64
    LSTM_LAYERS = 2
    LSTM_DROPOUT = 0.3
    LSTM_EPOCHS = 50
    LSTM_LR = 1e-3
    LSTM_BATCH = 64
    LSTM_PATIENCE = 10

    # CNN training
    CNN_EPOCHS = 80
    CNN_LR = 1e-3
    CNN_BATCH = 64
    CNN_PATIENCE = 15

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED = 42


# ============================================================
# 1. Build base bar features (for CNN and LSTM)
# ============================================================
def build_base_features(df_15min):
    """5 basic features for LSTM input."""
    close = df_15min['close'].values.astype(float)
    open_ = df_15min['open'].values.astype(float)
    high = df_15min['high'].values.astype(float)
    low = df_15min['low'].values.astype(float)
    volume = df_15min['volume'].values.astype(float)
    n = len(close)

    ret = np.zeros(n)
    ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-8)
    hl_range = (high - low) / (close + 1e-8)
    body = (close - open_) / (close + 1e-8)
    vol_ma = pd.Series(volume).rolling(20, min_periods=1).mean().values
    vol_ratio = np.log1p(np.clip(volume / (vol_ma + 1e-8), 0, 10))
    position = (close - low) / (high - low + 1e-8)

    features = np.stack([ret, hl_range, body, vol_ratio, position], axis=1)
    return np.nan_to_num(features, nan=0.0).astype(np.float32)


# ============================================================
# 2. Volatility predictor — Direct LSTM with HAR features
#    Adapted from Volatility V3 (75.7% DirAcc on 30min blocks)
#    Here: 15min bars, block_size=24 (6 hours, same as V3)
# ============================================================
def compute_volatility_features(df_15min, config):
    """
    Compute block-level realized volatility on 15min data.
    Block = 24 bars of 15min = 6 hours (same duration as V3's 12×30min).

    Returns per-bar arrays (block values broadcast to bars within block):
    - current_vol: current block's RV
    - vol_pred: predicted next block's RV (from Direct LSTM)

    Also returns block-level data for training.
    """
    close = df_15min['close'].values.astype(float)
    n = len(close)
    ret = np.zeros(n)
    ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-8)

    BLOCK_SIZE = 24  # 24 × 15min = 6 hours

    # Non-overlapping blocks
    n_blocks = n // BLOCK_SIZE
    usable = n_blocks * BLOCK_SIZE
    blocks_ret = ret[:usable].reshape(n_blocks, BLOCK_SIZE)

    # Block RV = std of returns within block
    block_rv = np.array([blocks_ret[i].std() for i in range(n_blocks)])
    block_agg_ret = np.array([blocks_ret[i].sum() for i in range(n_blocks)])

    # HAR features (Corsi 2009)
    rv_s = pd.Series(block_rv)
    rv_d = rv_s.values                                          # daily
    rv_w = rv_s.rolling(5, min_periods=1).mean().values         # weekly
    rv_m = rv_s.rolling(22, min_periods=1).mean().values        # monthly

    # Extra features
    rv_lag1 = np.zeros(n_blocks); rv_lag1[1:] = block_rv[:-1]
    rv_lag2 = np.zeros(n_blocks); rv_lag2[2:] = block_rv[:-2]
    rv_lag3 = np.zeros(n_blocks); rv_lag3[3:] = block_rv[:-3]
    rv_lag5 = np.zeros(n_blocks); rv_lag5[5:] = block_rv[:-5]
    rv_change = np.zeros(n_blocks); rv_change[1:] = block_rv[1:] - block_rv[:-1]
    rv_ratio = block_rv / (rv_s.rolling(10, min_periods=1).mean().values + 1e-10)
    vol_of_vol = rv_s.rolling(10, min_periods=1).std().values
    abs_block_ret = np.abs(block_agg_ret)

    # Stack features: (n_blocks, n_feat)
    block_features = np.stack([
        rv_d, rv_w, rv_m,
        rv_lag1, rv_lag2, rv_lag3, rv_lag5,
        rv_change, rv_ratio, vol_of_vol,
        block_agg_ret, abs_block_ret
    ], axis=1).astype(np.float32)

    # Target: next block RV
    block_target = np.zeros(n_blocks, dtype=np.float32)
    block_target[:-1] = block_rv[1:]

    # Broadcast block values to bar level
    current_vol = np.zeros(n, dtype=np.float32)
    for i in range(n_blocks):
        s, e = i * BLOCK_SIZE, (i + 1) * BLOCK_SIZE
        current_vol[s:e] = block_rv[i]
    # Fill tail
    if usable < n:
        current_vol[usable:] = block_rv[-1]

    return current_vol, block_features, block_target, block_rv, n_blocks, BLOCK_SIZE


class DirectVolLSTM(nn.Module):
    """Feature-based LSTM for volatility prediction (same arch as V3)."""
    def __init__(self, n_feat, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


def train_vol_predictor(base_features_unused, current_vol, block_features,
                        block_target, block_rv, n_blocks, BLOCK_SIZE,
                        train_end_idx, n_bars, config):
    """
    Train Direct LSTM on block-level HAR features → predict next block RV.
    Then broadcast predictions back to bar level.
    """
    from sklearn.preprocessing import StandardScaler

    SEQ_LEN = 20  # 20 blocks lookback
    n_feat = block_features.shape[1]

    # Train/test split at block level
    train_end_block = train_end_idx // BLOCK_SIZE
    if train_end_block < SEQ_LEN + 10 or train_end_block >= n_blocks - 5:
        print(f"    Vol: insufficient blocks, using current_vol only")
        return None, np.zeros(n_bars, dtype=np.float32)

    # Normalize
    sc_x = StandardScaler()
    sc_y = StandardScaler()
    feat_scaled = sc_x.fit_transform(block_features[:train_end_block])
    # Transform all blocks using train stats
    all_feat_scaled = sc_x.transform(block_features)
    target_scaled = sc_y.fit_transform(block_target[:train_end_block].reshape(-1, 1)).ravel()
    all_target_scaled = sc_y.transform(block_target.reshape(-1, 1)).ravel()

    # Build sequences
    def make_seq(X, y, sl, start, end):
        Xs, ys = [], []
        for i in range(start + sl, end):
            Xs.append(X[i - sl:i])
            ys.append(y[i])
        return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

    X_tr, y_tr = make_seq(all_feat_scaled, all_target_scaled, SEQ_LEN, 0, train_end_block)

    if len(X_tr) < 30:
        print(f"    Vol: too few sequences ({len(X_tr)})")
        return None, np.zeros(n_bars, dtype=np.float32)

    # Val split
    vn = max(int(len(X_tr) * 0.15), 1)
    X_v, y_v = X_tr[-vn:], y_tr[-vn:]
    X_tr, y_tr = X_tr[:-vn], y_tr[:-vn]

    # Train
    model = DirectVolLSTM(n_feat, hidden=64, layers=2, dropout=0.2).to(config.DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()

    ds = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr))
    dl = DataLoader(ds, batch_size=32, shuffle=True)
    Xv_t = torch.FloatTensor(X_v).to(config.DEVICE)
    yv_t = torch.FloatTensor(y_v).to(config.DEVICE)

    best_vl, best_st, pat = float('inf'), None, 0
    for ep in range(100):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xv_t), yv_t).item()
        if vl < best_vl:
            best_vl, best_st, pat = vl, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= 15:
                break

    if best_st:
        model.load_state_dict(best_st)
        model = model.to(config.DEVICE)

    # Predict ALL blocks
    model.eval()
    block_vol_pred = np.zeros(n_blocks, dtype=np.float32)
    with torch.no_grad():
        for i in range(SEQ_LEN, n_blocks):
            inp = torch.FloatTensor(all_feat_scaled[i - SEQ_LEN:i]).unsqueeze(0).to(config.DEVICE)
            p = model(inp).item()
            block_vol_pred[i] = max(sc_y.inverse_transform([[p]])[0, 0], 0)

    # Check DirAcc on test blocks
    test_blocks = slice(train_end_block, n_blocks - 1)
    actual_dir = block_target[test_blocks] > block_rv[test_blocks]
    pred_dir = block_vol_pred[test_blocks] > block_rv[test_blocks]
    n_test = min(len(actual_dir), len(pred_dir))
    if n_test > 0:
        da = np.mean(actual_dir[:n_test] == pred_dir[:n_test]) * 100
        print(f"    Vol LSTM DirAcc: {da:.1f}% (N={n_test} blocks)")

    # Broadcast block predictions to bar level
    vol_pred = np.zeros(n_bars, dtype=np.float32)
    for i in range(n_blocks):
        s, e = i * BLOCK_SIZE, min((i + 1) * BLOCK_SIZE, n_bars)
        vol_pred[s:e] = block_vol_pred[i]
    if n_blocks * BLOCK_SIZE < n_bars:
        vol_pred[n_blocks * BLOCK_SIZE:] = block_vol_pred[-1]

    return model, vol_pred


# ============================================================
# 3. Generate CNN probabilities for each bar
# ============================================================
def generate_cnn_probs(bottom_cnn, top_cnn, features_16, df_15min, config):
    """
    For each bar, compute CNN bottom and top reversal probability.
    Uses the trained Dual CNN models.
    """
    n = len(features_16)
    W = config.CNN_WINDOW

    bottom_probs = np.zeros(n, dtype=np.float32)
    top_probs = np.zeros(n, dtype=np.float32)

    # Check trend direction at each bar
    close = df_15min['close'].values.astype(float)
    trend_pct = config.TREND_PCT / 100.0
    LB = config.TREND_LOOKBACK

    # Batch process for efficiency
    down_indices = []
    up_indices = []
    down_windows = []
    up_windows = []

    for t in range(max(W, LB), n):
        p_now = close[t]
        p_past = close[t - LB]
        if p_now <= 0 or p_past <= 0:
            continue
        move = (p_now - p_past) / p_past

        window = features_16[t - W:t]

        if move < -trend_pct:
            down_indices.append(t)
            down_windows.append(window)
        elif move > trend_pct:
            up_indices.append(t)
            up_windows.append(window)

    # Batch predict bottom CNN
    if down_windows:
        X_down = np.array(down_windows, dtype=np.float32)
        bottom_cnn.eval()
        with torch.no_grad():
            for i in range(0, len(X_down), 1024):
                b = torch.FloatTensor(X_down[i:i+1024]).to(config.DEVICE)
                p = torch.sigmoid(bottom_cnn(b)).cpu().numpy()
                for j, idx in enumerate(down_indices[i:i+1024]):
                    bottom_probs[idx] = p[j]

    # Batch predict top CNN
    if up_windows:
        X_up = np.array(up_windows, dtype=np.float32)
        top_cnn.eval()
        with torch.no_grad():
            for i in range(0, len(X_up), 1024):
                b = torch.FloatTensor(X_up[i:i+1024]).to(config.DEVICE)
                p = torch.sigmoid(top_cnn(b)).cpu().numpy()
                for j, idx in enumerate(up_indices[i:i+1024]):
                    top_probs[idx] = p[j]

    return bottom_probs, top_probs


# ============================================================
# 4. Build enhanced TP dataset
# ============================================================
def build_tp_dataset_enhanced(df_15min, base_features, bottom_probs, top_probs,
                               current_vol, vol_pred, config):
    """
    Find turning points, build windows with base features + extra channels.

    Extra channels at each bar:
    - cnn_bottom_prob: probability of V-bottom reversal
    - cnn_top_prob: probability of ^-top reversal
    - current_vol: realized volatility
    - vol_pred: predicted future volatility

    Total features per bar: 5 (base) + 4 (extra) = 9
    """
    close = df_15min['close'].values.astype(float)
    high = df_15min['high'].values.astype(float)
    low = df_15min['low'].values.astype(float)
    timestamps = df_15min['timestamp'].values
    n = len(close)

    W = config.LSTM_WINDOW
    rev_thr = config.REVERSAL_THR / 100.0
    min_dur = config.MIN_DURATION // config.RESAMPLE_PERIOD
    pred_h = config.PRED_HORIZON // config.RESAMPLE_PERIOD

    # Find turning points using zigzag
    tps = find_turning_points_zigzag(close, high, low, rev_thr, min_dur)

    X_list, y_list, meta_list = [], [], []

    for tp_idx, tp_type in tps:
        if tp_idx - W < 0 or tp_idx + pred_h >= n:
            continue

        # Base features window
        base_win = base_features[tp_idx - W:tp_idx]  # (W, 5)

        # Extra features window
        bp_win = bottom_probs[tp_idx - W:tp_idx].reshape(-1, 1)  # (W, 1)
        tp_win = top_probs[tp_idx - W:tp_idx].reshape(-1, 1)     # (W, 1)
        cv_win = current_vol[tp_idx - W:tp_idx].reshape(-1, 1)   # (W, 1)
        vp_win = vol_pred[tp_idx - W:tp_idx].reshape(-1, 1)      # (W, 1)

        # Concatenate: (W, 9)
        combined = np.concatenate([base_win, bp_win, tp_win, cv_win, vp_win], axis=1)

        # Label: direction after turning point
        future_price = close[tp_idx + pred_h]
        current_price = close[tp_idx]
        if current_price <= 0:
            continue
        direction = 1 if future_price > current_price else 0

        X_list.append(combined)
        y_list.append(direction)
        meta_list.append({
            'tp_idx': tp_idx,
            'tp_type': tp_type,
            'ticker': '',  # filled later
            'timestamp': timestamps[tp_idx]
        })

    if len(X_list) == 0:
        return None, None, None

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    return X, y, meta_list


def find_turning_points_zigzag(close, high, low, rev_thr, min_dur):
    """Simple zigzag turning point detection."""
    n = len(close)
    tps = []

    if n < 10:
        return tps

    direction = 0  # 0=unknown, 1=up, -1=down
    last_high_idx = 0
    last_low_idx = 0
    last_high = close[0]
    last_low = close[0]

    for i in range(1, n):
        if direction == 0:
            if close[i] > last_high * (1 + rev_thr):
                direction = 1
                last_low_idx = 0
                last_high = close[i]
                last_high_idx = i
            elif close[i] < last_low * (1 - rev_thr):
                direction = -1
                last_high_idx = 0
                last_low = close[i]
                last_low_idx = i
            else:
                if close[i] > last_high:
                    last_high = close[i]
                    last_high_idx = i
                if close[i] < last_low:
                    last_low = close[i]
                    last_low_idx = i
        elif direction == 1:
            if close[i] > last_high:
                last_high = close[i]
                last_high_idx = i
            elif close[i] < last_high * (1 - rev_thr):
                if last_high_idx - (tps[-1][0] if tps else 0) >= min_dur:
                    tps.append((last_high_idx, 'top'))
                direction = -1
                last_low = close[i]
                last_low_idx = i
        elif direction == -1:
            if close[i] < last_low:
                last_low = close[i]
                last_low_idx = i
            elif close[i] > last_low * (1 + rev_thr):
                if last_low_idx - (tps[-1][0] if tps else 0) >= min_dur:
                    tps.append((last_low_idx, 'bottom'))
                direction = 1
                last_high = close[i]
                last_high_idx = i

    return tps


# ============================================================
# 5. Attention-LSTM Model
# ============================================================
class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out):
        scores = self.attn(lstm_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights


class AttnLSTM(nn.Module):
    def __init__(self, n_features, hidden_dim=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_dim, n_layers,
                           batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.attention = AttentionLayer(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, _ = self.attention(lstm_out)
        return self.head(context).squeeze(-1)


# ============================================================
# 6. Train Attn-LSTM
# ============================================================
def train_attn_lstm(X_tr, y_tr, X_val, y_val, config, n_features, label=""):
    model = AttnLSTM(n_features, config.LSTM_HIDDEN, config.LSTM_LAYERS,
                     config.LSTM_DROPOUT).to(config.DEVICE)

    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=config.LSTM_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', 0.5, patience=3)

    tds = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr.astype(np.float32)))
    vds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val.astype(np.float32)))
    tl = DataLoader(tds, batch_size=config.LSTM_BATCH, shuffle=True)
    vl = DataLoader(vds, batch_size=config.LSTM_BATCH)

    best_vl, pc, best_st = float('inf'), 0, None
    for ep in range(config.LSTM_EPOCHS):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        vloss = 0
        with torch.no_grad():
            for xb, yb in vl:
                xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
                vloss += crit(model(xb), yb).item() * len(yb)
        vloss /= len(vds); sched.step(vloss)

        if vloss < best_vl:
            best_vl = vloss
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pc = 0
        else:
            pc += 1
        if (ep + 1) % 10 == 0:
            print(f"      [{label}] Epoch {ep+1}: val_loss={vloss:.4f}")
        if pc >= config.LSTM_PATIENCE:
            print(f"      [{label}] Early stop at {ep+1}")
            break

    if best_st:
        model.load_state_dict(best_st)
        model = model.to(config.DEVICE)
    print(f"      [{label}] Best val_loss: {best_vl:.4f}")
    return model


# ============================================================
# 7. Evaluate DA
# ============================================================
def eval_da(model, X_test, y_test, config, label=""):
    model.eval()
    with torch.no_grad():
        out = []
        for i in range(0, len(X_test), 1024):
            b = torch.FloatTensor(X_test[i:i+1024]).to(config.DEVICE)
            out.append(torch.sigmoid(model(b)).cpu().numpy())
    probs = np.concatenate(out)
    preds = (probs > 0.5).astype(int)

    da = np.mean(preds == y_test) * 100
    n = len(y_test)
    z = (da / 100 - 0.5) / np.sqrt(0.25 / n)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''

    print(f"  {label}: DA={da:.1f}% (N={n}, z={z:+.2f}, p={p:.4f}) {sig}")
    return da, n, z, p


# ============================================================
# 8. Main — A/B comparison
# ============================================================
def run_enhanced_lstm():
    config = EnhancedConfig()
    set_seed(config.SEED)

    print(f"\n{'='*70}")
    print(f"  ENHANCED Attn-LSTM: Base vs Base+CNN+Vol")
    print(f"  TP params: rev={config.REVERSAL_THR}%, dur={config.MIN_DURATION}min, "
          f"horizon={config.PRED_HORIZON}min")
    print(f"{'='*70}")

    # ── Step 1: Train Dual CNNs ──
    print(f"\n[1] Training Dual CNNs...")

    down_tr_X, down_tr_y, down_v_X, down_v_y = [], [], [], []
    up_tr_X, up_tr_y, up_v_X, up_v_y = [], [], [], []

    # Store per-ticker data for later
    ticker_data = {}

    for ticker in config.TICKERS:
        data = load_ticker_data(ticker, config)
        if data is None:
            continue
        df = data['df']
        train_end_ts = data['train_end_ts']

        df_15min = resample_to_15min(df, config)
        features_16 = build_features_dual(df_15min)
        base_feat = build_base_features(df_15min)
        current_vol, block_features, block_target, block_rv, n_blocks, BLOCK_SIZE = \
            compute_volatility_features(df_15min, config)

        train_mask = df_15min['timestamp'] <= train_end_ts
        train_end_idx = int(train_mask.sum())

        # CNN training data
        positions = find_trend_and_label(df_15min, config)
        pos_train_down = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'down']
        pos_train_up = [p for p in positions if p['bar_idx'] < train_end_idx and p['trend_dir'] == 'up']

        if len(pos_train_down) > 30:
            X_d, y_d = build_dataset(features_16, pos_train_down, config)
            vs = max(int(len(X_d) * 0.15), 1)
            down_tr_X.append(X_d[:-vs]); down_tr_y.append(y_d[:-vs])
            down_v_X.append(X_d[-vs:]); down_v_y.append(y_d[-vs:])

        if len(pos_train_up) > 30:
            X_u, y_u = build_dataset(features_16, pos_train_up, config)
            vs = max(int(len(X_u) * 0.15), 1)
            up_tr_X.append(X_u[:-vs]); up_tr_y.append(y_u[:-vs])
            up_v_X.append(X_u[-vs:]); up_v_y.append(y_u[-vs:])

        ticker_data[ticker] = {
            'df_15min': df_15min,
            'features_16': features_16,
            'base_feat': base_feat,
            'current_vol': current_vol,
            'block_features': block_features,
            'block_target': block_target,
            'block_rv': block_rv,
            'n_blocks': n_blocks,
            'BLOCK_SIZE': BLOCK_SIZE,
            'train_end_idx': train_end_idx,
            'train_end_ts': train_end_ts,
        }
        print(f"  {ticker}: {len(df_15min)} bars, {n_blocks} vol blocks, train_end={train_end_idx}")

    # Train CNNs
    Xd_tr = np.concatenate(down_tr_X); yd_tr = np.concatenate(down_tr_y)
    Xd_v = np.concatenate(down_v_X); yd_v = np.concatenate(down_v_y)
    Xu_tr = np.concatenate(up_tr_X); yu_tr = np.concatenate(up_tr_y)
    Xu_v = np.concatenate(up_v_X); yu_v = np.concatenate(up_v_y)

    print(f"  Bottom CNN: train={len(Xd_tr)}, val={len(Xd_v)}")
    print(f"  Top CNN: train={len(Xu_tr)}, val={len(Xu_v)}")

    set_seed(config.SEED)
    bottom_cnn = train_specialized(Xd_tr, yd_tr, Xd_v, yd_v, config, "BOTTOM")
    set_seed(config.SEED)
    top_cnn = train_specialized(Xu_tr, yu_tr, Xu_v, yu_v, config, "TOP")

    # ── Step 2: Train vol predictors + generate CNN probs per ticker ──
    print(f"\n[2] Generating CNN probs & vol predictions per ticker...")

    for ticker, td in ticker_data.items():
        # Vol predictor (Direct LSTM with HAR features)
        _, vol_pred = train_vol_predictor(
            td['base_feat'], td['current_vol'],
            td['block_features'], td['block_target'],
            td['block_rv'], td['n_blocks'], td['BLOCK_SIZE'],
            td['train_end_idx'], len(td['df_15min']), config)
        td['vol_pred'] = vol_pred

        # CNN probs
        bp, tp = generate_cnn_probs(
            bottom_cnn, top_cnn, td['features_16'], td['df_15min'], config)
        td['bottom_probs'] = bp
        td['top_probs'] = tp
        print(f"  {ticker}: CNN probs generated, vol predictor trained")

    # ── Step 3: Build TP datasets ──
    print(f"\n[3] Building turning point datasets...")

    # A) Base features only (5 channels)
    base_tr_X, base_tr_y = [], []
    base_te_X, base_te_y = [], []
    base_te_meta = []

    # B) Enhanced features (9 channels)
    enh_tr_X, enh_tr_y = [], []
    enh_te_X, enh_te_y = [], []
    enh_te_meta = []

    for ticker, td in ticker_data.items():
        df_15min = td['df_15min']
        base_feat = td['base_feat']
        train_end_ts = td['train_end_ts']

        # Build enhanced dataset (9 features)
        X_enh, y_enh, meta_enh = build_tp_dataset_enhanced(
            df_15min, base_feat,
            td['bottom_probs'], td['top_probs'],
            td['current_vol'], td['vol_pred'], config)

        if X_enh is None or len(X_enh) < 10:
            print(f"  {ticker}: too few TPs, skipping")
            continue

        # Base dataset: first 5 features only
        X_base = X_enh[:, :, :5]

        # Fill ticker in meta
        for m in meta_enh:
            m['ticker'] = ticker

        # Split by timestamp
        train_mask = np.array([m['timestamp'] <= np.datetime64(train_end_ts)
                               for m in meta_enh])

        n_tr = train_mask.sum()
        n_te = len(train_mask) - n_tr

        if n_tr > 10:
            base_tr_X.append(X_base[train_mask])
            base_tr_y.append(y_enh[train_mask])
            enh_tr_X.append(X_enh[train_mask])
            enh_tr_y.append(y_enh[train_mask])

        if n_te > 5:
            base_te_X.append(X_base[~train_mask])
            base_te_y.append(y_enh[~train_mask])
            base_te_meta.extend([m for m, mask in zip(meta_enh, ~train_mask) if mask])
            enh_te_X.append(X_enh[~train_mask])
            enh_te_y.append(y_enh[~train_mask])
            enh_te_meta.extend([m for m, mask in zip(meta_enh, ~train_mask) if mask])

        print(f"  {ticker}: {n_tr} train TPs, {n_te} test TPs")

    # Concatenate
    Xb_tr = np.concatenate(base_tr_X); yb_tr = np.concatenate(base_tr_y)
    Xb_te = np.concatenate(base_te_X); yb_te = np.concatenate(base_te_y)
    Xe_tr = np.concatenate(enh_tr_X); ye_tr = np.concatenate(enh_tr_y)
    Xe_te = np.concatenate(enh_te_X); ye_te = np.concatenate(enh_te_y)

    # Validation split from training
    n_val = max(int(len(Xb_tr) * 0.15), 1)
    Xb_v, yb_v = Xb_tr[-n_val:], yb_tr[-n_val:]
    Xb_tr, yb_tr = Xb_tr[:-n_val], yb_tr[:-n_val]
    Xe_v, ye_v = Xe_tr[-n_val:], ye_tr[-n_val:]
    Xe_tr, ye_tr = Xe_tr[:-n_val], ye_tr[:-n_val]

    print(f"\n  Base: train={len(Xb_tr)}, val={len(Xb_v)}, test={len(Xb_te)}")
    print(f"  Enhanced: train={len(Xe_tr)}, val={len(Xe_v)}, test={len(Xe_te)}")
    print(f"  Test direction balance: {yb_te.mean()*100:.1f}% up")

    # ── Step 4: Train and compare ──
    print(f"\n[4] Training Attn-LSTM models...")

    # A) Baseline: 5 features
    print(f"\n  --- Model A: Base features (5ch) ---")
    set_seed(config.SEED)
    model_base = train_attn_lstm(Xb_tr, yb_tr, Xb_v, yb_v, config,
                                  n_features=5, label="BASE")

    # B) Enhanced: 9 features
    print(f"\n  --- Model B: Enhanced features (9ch = 5 base + CNN + Vol) ---")
    set_seed(config.SEED)
    model_enh = train_attn_lstm(Xe_tr, ye_tr, Xe_v, ye_v, config,
                                 n_features=9, label="ENHANCED")

    # ── Step 5: Evaluate ──
    print(f"\n{'='*70}")
    print(f"  RESULTS")
    print(f"{'='*70}")

    da_base, n_base, z_base, p_base = eval_da(
        model_base, Xb_te, yb_te, config, "Baseline (5 features)")
    da_enh, n_enh, z_enh, p_enh = eval_da(
        model_enh, Xe_te, ye_te, config, "Enhanced (9 features)")

    # Per-ticker
    print(f"\n  ── PER-TICKER ──")
    for ticker in config.TICKERS:
        mask = np.array([m['ticker'] == ticker for m in enh_te_meta])
        if mask.sum() < 20:
            continue

        Xb_t = Xb_te[mask]; yb_t = yb_te[mask]
        Xe_t = Xe_te[mask]; ye_t = ye_te[mask]

        model_base.eval(); model_enh.eval()
        with torch.no_grad():
            pb = torch.sigmoid(model_base(torch.FloatTensor(Xb_t).to(config.DEVICE))).cpu().numpy()
            pe = torch.sigmoid(model_enh(torch.FloatTensor(Xe_t).to(config.DEVICE))).cpu().numpy()

        da_b = np.mean((pb > 0.5).astype(int) == yb_t) * 100
        da_e = np.mean((pe > 0.5).astype(int) == ye_t) * 100
        diff = da_e - da_b
        print(f"  {ticker:>6s}: base={da_b:.1f}% → enh={da_e:.1f}% (Δ={diff:+.1f}%, N={mask.sum()})")

    print(f"\n  Original baseline (separate training): 53.2% (N=2694, p=0.0008)")
    print(f"  Shared CNN v2 downtrend thr=0.60: 78.7% (N=94)")
    print(f"  Top CNN thr=0.58: 56.9% (N=1012)")


run_enhanced_lstm()


  ENHANCED Attn-LSTM: Base vs Base+CNN+Vol
  TP params: rev=1.0%, dur=60min, horizon=45min

[1] Training Dual CNNs...
  Resampled: 574,498 (1min) → 43,340 (15min)
  AAPL: 43340 bars, 1805 vol blocks, train_end=34931
  Resampled: 497,673 (1min) → 42,907 (15min)
  MSFT: 42907 bars, 1787 vol blocks, train_end=34470
  Resampled: 403,977 (1min) → 39,056 (15min)
  GOOGL: 39056 bars, 1627 vol blocks, train_end=31991
  Resampled: 372,079 (1min) → 38,129 (15min)
  GOOG: 38129 bars, 1588 vol blocks, train_end=30830
  Resampled: 511,102 (1min) → 42,662 (15min)
  NVDA: 42662 bars, 1777 vol blocks, train_end=35013
  Resampled: 615,969 (1min) → 43,399 (15min)
  TSLA: 43399 bars, 1808 vol blocks, train_end=34930
  Resampled: 532,419 (1min) → 43,233 (15min)
  SPY: 43233 bars, 1801 vol blocks, train_end=34380
  Resampled: 560,063 (1min) → 43,332 (15min)
  QQQ: 43332 bars, 1805 vol blocks, train_end=35132
  Bottom CNN: train=33235, val=5861
  Top CNN: train=35032, val=6176
      [BOTTOM] Epoch 10: val_